In [ ]:
import os
import numpy as np
import jax
import jax.numpy as jnp
from jax.sharding import AxisType
from qiskit.quantum_info import SparsePauliOp
from rqutils.sqd import sqd
from rqutils.paulis.symplectic import PauliSumXZ

os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
jax.config.update('jax_enable_x64', True)

In [ ]:
NQ = 12
NSAMPLE = 100_000
NTERM = 100
NEXP = 10

In [ ]:
if (ndev := jax.device_count()) > 1:
    jax.set_mesh(jax.make_mesh((ndev,), ('x',), (AxisType.Explicit,)))

In [ ]:
rng = np.random.default_rng()

for iexp in range(NEXP):
    print('Experiment', iexp)
    print('  Generating states..')
    states = rng.choice(2, size=(NSAMPLE, NQ)).astype(np.uint8)

    paulis = [''.join('IXYZ'[ip] for ip in row) for row in rng.choice(4, size=(NTERM, NQ))]
    coeffs = rng.uniform(-1., 1., size=NTERM)
    hamiltonian = SparsePauliOp(paulis, coeffs)

    print('  Testing sqd..')
    eigval, eigvec, basis_states = sqd(hamiltonian, states, return_eigvec=True)

    print('   Computing reference..')
    hmat = jnp.array(hamiltonian.to_matrix())
    states_u = jnp.unique(states, axis=0)
    indices = jnp.sum(states_u * (1 << jnp.arange(NQ)[None, ::-1]), axis=1)
    hproj = hmat[indices][:, indices]
    eigvals_ref, eigvecs_ref = jnp.linalg.eigh(hproj)

    print('  eigvals close:', jnp.isclose(eigval, eigvals_ref[0]))
    print('  eigvecs overlap:', jnp.abs(jnp.vdot(eigvecs_ref[:, 0], eigvec)))